# Replication run: second seed + out-of-range shifts (Kaggle)
Trains A_baseline, G_colorjitter, H_stainaug and C_stainOD+consist again with **seed 43** (new slide folds) and tests them under
shifts **stronger than any augmentation saw**: brightness x0.65 / x0.5, strong colour casts (wb0.2), stain scaling x2.0 / x0.4.
Needs the same two Input datasets (`physics-paper`, `physics-code`), GPU T4 x2 and Internet On. Expect roughly 2-3 h.

## 1. GPU check

In [ ]:
import os, subprocess, torch
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Session options > Accelerator > GPU T4 x2, then re-run"

## 2. Setup: find files, unpack patches, patch the colour-cast test

In [ ]:
import os, shutil, zipfile, time, re
INPUT, WORK, CODE, TMP = "/kaggle/input", "/kaggle/working", "/kaggle/working/code", "/kaggle/temp"
os.makedirs(CODE, exist_ok=True); os.makedirs(TMP, exist_ok=True)

def find_file(name):
    stem, ext = os.path.splitext(name)
    pat = re.compile(rf"^{re.escape(stem)}( \((\d+)\))?{re.escape(ext)}$")
    hits = []
    for dp, _, fs in os.walk(INPUT):
        for f in fs:
            m = pat.match(f)
            if m:
                hits.append((int(m.group(2) or 0), os.path.join(dp, f)))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {INPUT} - did you add both datasets as Input?")
    return max(hits)[1]

for f in ["ovarian_pipeline.py", "novelty_utils.py", "novel_model.py",
          "histo_utils.py", "histo_model.py", "test_histo_utils.py"]:
    src = find_file(f)
    shutil.copy(src, f"{CODE}/{f}")
    print("code:", os.path.basename(src), "->", f)

# allow "wb<s>" (stronger white-balance cast); "wb" keeps its old meaning. Skipped if already present.
p = f"{CODE}/histo_utils.py"
txt = open(p).read()
old = '''    elif kind == "wb":
        out = img.astype(np.float32) * np.array([1.0, 0.9, 0.8], np.float32)'''
new = '''    elif kind.startswith("wb"):
        s = float(kind[2:]) if len(kind) > 2 else 0.1
        out = img.astype(np.float32) * np.array([1.0, 1 - s, 1 - 2 * s], np.float32)'''
if 'kind.startswith("wb")' not in txt:
    assert old in txt, "histo_utils.py differs from the expected version"
    open(p, "w").write(txt.replace(old, new))
print("wb patch ok")

t = time.time()
UBC = f"{TMP}/ubc"
zips = [os.path.join(dp, f) for dp, _, fs in os.walk(INPUT) for f in fs if f == "ubc_patches.zip"]
if zips:
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(TMP)
else:
    man = find_file("patch_manifest.csv")
    shutil.copytree(os.path.dirname(os.path.dirname(man)), UBC, dirs_exist_ok=True)
assert os.path.exists(f"{UBC}/patches/patch_manifest.csv"), f"unexpected layout under {UBC}"
print(f"patches ready: {sum(len(fs) for _, _, fs in os.walk(UBC))} files in {time.time() - t:.0f}s")
os.chdir(CODE)

## 3. Quick check that the new conditions work (about 1 minute)

In [ ]:
!python -W ignore test_histo_utils.py 2>&1 | tail -3

## 4. Main run (seed 43, 4 variants). If the session stops, re-run this cell: `--resume` continues after the last finished fold

In [ ]:
OUT = f"{WORK}/results_seed43"
!python histo_model.py --ubc-dir {UBC} --max-patches-per-slide 16 --folds 5 --epochs 10 --seed 43 --variants A_baseline G_colorjitter H_stainaug C_stainOD+consist --conditions clean clean_sub bright0.8 bright0.65 bright0.5 wb wb0.2 hscale1.3 hscale2.0 escale0.7 escale0.4 jpeg30 noise --workers 2 --dl-workers 2 --resume --out {OUT}

## 5. Bootstrap CIs and zip of the results

In [ ]:
import sys, shutil, numpy as np
sys.path.insert(0, CODE)
import histo_utils as hu
BRING = f"{WORK}/bring_back_seed43"; os.makedirs(BRING, exist_ok=True)
try:
    sp = hu.load_slide_predictions(f"{OUT}/slide_predictions.csv")
    vs = [v for v in ["A_baseline", "G_colorjitter", "H_stainaug", "C_stainOD+consist"] if v in set(sp.variant)]
    text = "\n\n".join([hu.bootstrap_table_md(sp, vs, ref="A_baseline"), hu.bootstrap_table_md(sp, vs, ref="H_stainaug")])
    open(f"{BRING}/bootstrap_seed43.md", "w", encoding="utf-8").write(text)
    print(text)
except Exception as e:
    print("bootstrap step failed:", repr(e))
print(open(f"{OUT}/tables.md", encoding="utf-8").read())
shutil.copytree(OUT, f"{BRING}/results_seed43", dirs_exist_ok=True)
shutil.make_archive(f"{WORK}/results_seed43_all", "zip", WORK, "bring_back_seed43")
print("zip MB:", round(os.path.getsize(f"{WORK}/results_seed43_all.zip") / 1e6, 1))